# GRU Model for Text Classification

This notebook handles the entire pipeline for training a GRU model on text data:
1. Data loading and preprocessing
2. Text tokenization and sequence creation
3. Word embeddings
4. Hyperparameter tuning
5. Model training and evaluation
6. Model saving


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, initializers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import re
import string
import nltk
from nltk.tokenize import word_tokenize
import keras_tuner as kt
import os

nltk.download('punkt', quiet=True)

np.random.seed(2025)
tf.random.set_seed(2025)


## Data Preparation

In [ ]:
print("Loading dataset...")
df = pd.read_csv("../../datasets/custom_dataset.csv", sep="\t")
print(f"Dataset shape: {df.shape}")
df.head()


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    
    text = re.sub(r"https?://\S+|www\.\S+|\S+@\S+\.\S+", "", text)
    
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

df['cleaned_text'] = df['Text'].apply(clean_text)

df[['Text', 'cleaned_text']].sample(5)

labels = df['Label'].apply(lambda x: 0 if x == 'Human' else 1).values

print(f"Label distribution: {pd.Series(labels).value_counts().to_dict()}")

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['cleaned_text'].values, labels, test_size=0.3, random_state=2025, stratify=labels
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=2025, stratify=temp_labels
)

print(f"Training set: {len(train_texts)} samples")
print(f"Validation set: {len(val_texts)} samples")
print(f"Test set: {len(test_texts)} samples")


In [ ]:
max_words = 2500
max_seq_length = 128

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

vocab_size = min(max_words, len(tokenizer.word_index) + 1)
print(f"Vocabulary size: {vocab_size}")

train_sequences = tokenizer.texts_to_sequences(train_texts)
val_sequences = tokenizer.texts_to_sequences(val_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

train_padded = pad_sequences(train_sequences, maxlen=max_seq_length, padding='post', truncating='post')
val_padded = pad_sequences(val_sequences, maxlen=max_seq_length, padding='post', truncating='post')
test_padded = pad_sequences(test_sequences, maxlen=max_seq_length, padding='post', truncating='post')

print(f"Training sequences shape: {train_padded.shape}")
print(f"Validation sequences shape: {val_padded.shape}")
print(f"Test sequences shape: {test_padded.shape}")


## Model Development

In [ ]:
def build_gru_model(hp):
    model = keras.Sequential()
    
    embedding_dim = hp.Int('embedding_dim', 64, 256, step=64)
    model.add(layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_seq_length,
        embeddings_initializer=initializers.GlorotUniform(seed=2025)
    ))
    
    model.add(layers.SpatialDropout1D(
        hp.Float('spatial_dropout', 0.1, 0.5, step=0.1)
    ))
    
    if hp.Boolean('use_conv'):
        model.add(layers.Conv1D(
            filters=hp.Int('conv_filters', 32, 128, step=32),
            kernel_size=5,
            padding='same',
            activation='relu'
        ))
    
    model.add(layers.Bidirectional(layers.GRU(
        units=hp.Int('gru_units_1', 32, 256, step=32),
        return_sequences=True,
        dropout=hp.Float('dropout_1', 0.1, 0.5, step=0.1),
        recurrent_dropout=hp.Float('recurrent_dropout_1', 0.1, 0.5, step=0.1)
    )))
    
    model.add(layers.Bidirectional(layers.GRU(
        units=hp.Int('gru_units_2', 32, 128, step=32),
        dropout=hp.Float('dropout_2', 0.1, 0.5, step=0.1),
        recurrent_dropout=hp.Float('recurrent_dropout_2', 0.1, 0.5, step=0.1)
    )))
    
    model.add(layers.Dense(
        units=hp.Int('dense_units', 16, 64, step=16),
        activation='relu'
    ))
    model.add(layers.Dropout(
        hp.Float('dense_dropout', 0.1, 0.5, step=0.1)
    ))
    
    model.add(layers.Dense(1, activation='sigmoid'))
    
    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')
        ),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(), keras.metrics.Precision(), keras.metrics.Recall()]
    )
    
    return model

tuner = kt.Hyperband(
    build_gru_model,
    objective='val_accuracy',
    max_epochs=10,
    factor=3,
    directory='../../tuner_results',
    project_name='gru_model'
)

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=2,
    restore_best_weights=True
)

tuner.search(
    train_padded, train_labels,
    epochs=25,
    validation_data=(val_padded, val_labels),
    callbacks=[early_stopping]
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"Best hyperparameters: {best_hps.values}")


In [ ]:
model = tuner.hypermodel.build(best_hps)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('../../trained_models/tensorflow/gru_model.h5', monitor='val_loss', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

history = model.fit(
    train_padded, train_labels,
    validation_data=(val_padded, val_labels),
    epochs=25,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


## Model Evaluation

In [ ]:
model = keras.models.load_model('../../trained_models/tensorflow/gru_model.h5')

test_loss, test_acc, test_auc, test_precision, test_recall = model.evaluate(test_padded, test_labels)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(f"Test precision: {test_precision:.4f}")
print(f"Test recall: {test_recall:.4f}")

y_pred_prob = model.predict(test_padded)
y_pred = (y_pred_prob > 0.5).astype(int)

plt.figure(figsize=(8, 6))
cm = confusion_matrix(test_labels, y_pred)
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=['Human', 'AI'],
    yticklabels=['Human', 'AI']
)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print("Classification Report:")
print(classification_report(test_labels, y_pred, target_names=['Human', 'AI']))

fpr, tpr, _ = roc_curve(test_labels, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()


## Model Deployment

In [ ]:
model.save('../../trained_models/tensorflow/gru_model.h5')
print("Model saved to '../../trained_models/tensorflow/gru_model.h5'")

import pickle

with open('../../trained_models/tensorflow/gru_tokenizer.pkl', 'wb') as f:
    pickle.dump({
        'tokenizer': tokenizer,
        'max_seq_length': max_seq_length,
        'clean_text': clean_text
    }, f)
print("Tokenizer saved to '../../trained_models/tensorflow/gru_tokenizer.pkl'")

def predict_text(text, model, preprocessor):
    cleaned_text = preprocessor['clean_text'](text)
    
    sequence = preprocessor['tokenizer'].texts_to_sequences([cleaned_text])
    padded = pad_sequences(sequence, maxlen=preprocessor['max_seq_length'], padding='post', truncating='post')
    
    prediction = model.predict(padded)[0][0]
    
    return {
        'probability': float(prediction),
        'prediction': 'AI' if prediction > 0.5 else 'Human'
    }

loaded_model = keras.models.load_model('../../trained_models/tensorflow/gru_model.h5')
with open('../../trained_models/tensorflow/gru_tokenizer.pkl', 'rb') as f:
    loaded_preprocessor = pickle.load(f)

sample_text = "This is a sample text to test the model."
result = predict_text(sample_text, loaded_model, loaded_preprocessor)
print(f"Sample text: '{sample_text}'")
print(f"Prediction: {result['prediction']}")
